In [305]:
import pypsa
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

from synthseer import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Create network

In [313]:
model = pypsa.Network()
model.add("Carrier", "AC")

# Key dimensions of optimization
nzones = 1
nyears = 1
rez = 1 # resolution in hours


# Other user specified inputs
year_step = 1
start_year = 2025
end_year = start_year + nyears*year_step


# Derived inputs
years = [y for y in range(start_year, end_year, year_step)]

set_snapshots(model, years, rez)
set_obj_weights(model, years)

load_ts = synth_load_ts(start=str(start_year), end=str(end_year), freq=f"{rez}h")
re_ts = synth_re_ts(start=str(start_year), end=str(end_year), freq=f"{rez}h")

# The index needs to match the structure of snapshots
load_ts.index = model.snapshots
re_ts.index = model.snapshots

gens = ["pv", "wind", "gas"]
existing_cap = get_synth_caps(np.arange(nzones), gens, pd.DataFrame(index=gens, data=[100, 200, 500]))
starting_load = get_synth_load(np.arange(nzones), 600)

# get synthetic capital and marginal costs for generators
cap, marg = synth_cost_dfs(np.array(years))

# add buses
add_buses(model, nzones)

# add existing solar, wind, gas in each zone
add_gen_current(model, re_ts["pv_pu"], marg.loc[start_year, "pv"], existing_cap["pv"], name="pv")
add_gen_current(model, re_ts["wind_pu"], marg.loc[start_year, "wind"], existing_cap["wind"], name="wind")
add_gen_current(model, None, marg.loc[start_year, "gas"], existing_cap["gas"], name="gas")

# add build options
for year in years:
    # solar
    add_gen_options(model, year, re_ts["pv_pu"], cap.loc[year]["pv"], marg.loc[year]["pv"], name="pv")
    # wind
    add_gen_options(model, year, re_ts["wind_pu"], cap.loc[year]["wind"], marg.loc[year]["wind"], name="wind")
    # gas
    add_gen_options(model, year, None, cap.loc[year]["gas"], marg.loc[year]["gas"], name="gas")

# add expensive load shedding
add_load_shed(model, starting_load, load_ts["load_pu"])

# add load
add_load(model, starting_load, load_ts["load_pu"])

In [314]:
model.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████████| 2/2 [00:00<00:00, 18.86it/s]
INFO:linopy.io: Writing time: 1.16s


Running HiGHS 1.12.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-48giucni has 131403 rows; 61323 cols; 205953 nonzeros
Coefficient ranges:
  Matrix  [2e-04, 1e+00]
  Cost    [3e+00, 2e+05]
  Bound   [0e+00, 0e+00]
  RHS     [4e-02, 6e+02]
Presolving model
25183 rows, 41609 cols, 74452 nonzeros  0s
Dependent equations search running on 8760 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.01s (limit = 1000.00s)
25183 rows, 41609 cols, 74452 nonzeros  0s
Presolve reductions: rows 25183(-106220); columns 41609(-19714); nonzeros 74452(-131501) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 8760(1.31166e+06) 0s
      23710     1.8975602498e+08 Pr: 0(0) 1s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-48giucni
Model statu

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 61323 primals, 131403 duals
Objective: 1.90e+08
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper were not assigned to the network.


('ok', 'optimal')

In [315]:
model.model

Linopy LP model

Variables:
----------
 * Generator-p_nom (name)
 * Generator-p (snapshot, period, timestep, name)

Constraints:
------------
 * Generator-ext-p_nom-lower (name)
 * Generator-fix-p-lower (snapshot, period, timestep, name)
 * Generator-fix-p-upper (snapshot, period, timestep, name)
 * Generator-ext-p-lower (snapshot, period, timestep, name)
 * Generator-ext-p-upper (snapshot, period, timestep, name)
 * Bus-nodal_balance (name, snapshot, period, timestep)

Status:
-------
ok

In [312]:
model.statistics()

,,Optimal Capacity,Installed Capacity,Supply,Withdrawal,Energy Balance,Transmission,Capacity Factor,Curtailment,Capital Expenditure,Operational Expenditure,Revenue,Market Value
,,2025,2025,2025,2025,2025,0,2025,2025,2025,2025,2025,2025
Generator,-,1400.55479,1395.11431,474917.75898,NaN,474917.75898,0.0,0.309674,799362.47208,1.088096e+06,2.795990e+07,1.355799e+08,285.480737
Load,-,0.00000,0.00000,NaN,474917.75898,-474917.75898,0.0,NaN,0.00000,0.000000e+00,0.000000e+00,-1.355799e+08,NaN
